# 02 - Market data: acquisition and data-health audit

**Question:** Is the Binance BTCUSDT 1-minute data complete, consistent and correctly timestamped, and what must be treated before research?

**Why:** Every later conclusion inherits the data's problems. Known traps: the switch from millisecond to microsecond timestamps on 2025-01-01, exchange-downtime gaps, revised archive files and duplicate or conflicting rows.

**Method:** `src/quality.py` checks schema, timestamps, frequency, OHLC consistency, volume, duplicates, missing values and outliers. Treatments are rule-based and fixed in advance:
- exact duplicate rows are kept once;
- conflicting duplicates and impossible candles become missing;
- gaps are never filled;
- extreme but consistent moves are kept and flagged.

Data must already be downloaded; see `data/README.md`.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "scripts"))
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
import run_pipeline as rp
from src import config
T, F = config.TABLES_DIR, config.FIGURES_DIR
SYMBOL = config.SYMBOL
# False (default): display the outputs saved by scripts/run_pipeline.py.
# True re-computes the stage. Do NOT re-run 'validation' or 'test' casually:
# validation resets the experiment log and every test run adds a TEST row.
RUN_STAGE = False
def show(name, **kw):
    display(Markdown(f"**{name}**")); display(pd.read_csv(T / name, **kw))
def fig(name):
    display(Image(filename=str(F / name)))
def results(section):
    return json.loads((config.REPORTS_DIR / "results.json").read_text()).get(section, {})

In [2]:
if RUN_STAGE:
    rp.stage_audit(SYMBOL)

In [3]:
show(SYMBOL + '_audit_dq_table.csv')

**BTCUSDT_audit_dq_table.csv**

,check,result,severity,action
0,Rows / columns,"4,789,279 rows, 16 cols",info,NaN
1,Schema,missing cols: none; unexpected: none,info,Stop if any expected column is missing
2,Timestamp units,"89 files ms, 21 files us; unexpected unit in 0...",info,Unit detected per file and converted to UTC da...
3,Timestamp alignment,21602 not on minute boundary; 18 with close_ti...,medium,Misaligned rows reported; grid uses open_time
4,Rows outside their file's month,0,info,Reported
5,Ordering within files,0 files not time-ordered,info,Data sorted by open_time
6,Missing values (cells),0,info,Minute treated as missing
7,Duplicate rows (exact),0,info,Keep one copy
8,Duplicate timestamps,0 rows,info,See conflicting
9,Conflicting duplicates,0 timestamps,info,Minute set to missing


In [4]:
show(SYMBOL + '_audit_treatments.csv')

**BTCUSDT_audit_treatments.csv**

,treatment,count
0,drop exact duplicate rows,0
1,conflicting duplicate timestamps -> missing,0
2,impossible / incomplete candles -> missing,0
3,minutes missing on regular grid (left as NaN),30163


In [5]:
show(SYMBOL + '_audit_missing_by_year.csv')

**BTCUSDT_audit_missing_by_year.csv**

,year,expected_minutes,missing_minutes,missing_pct
0,2017,197040,20897,10.605461
1,2018,525600,5177,0.984970
2,2019,525600,1764,0.335616
3,2020,527040,1252,0.237553
4,2021,525600,993,0.188927
5,2022,525600,0,0.000000
6,2023,525600,80,0.015221
7,2024,527040,0,0.000000
8,2025,525600,0,0.000000
9,2026,393120,0,0.000000


In [6]:
show(SYMBOL + '_audit_gaps_top25.csv')

**BTCUSDT_audit_gaps_top25.csv**

,gap_start,gap_end,missing_minutes,off_grid_rows_inside,kind
0,2017-12-04 06:01:00+00:00,2017-12-18 10:13:00+00:00,20413,20400,"phase-shifted candles (data exist, off grid)"
1,2018-02-08 00:29:00+00:00,2018-02-10 06:14:00+00:00,3226,1201,mixed: outage + phase-shifted candles
2,2018-06-26 02:00:00+00:00,2018-06-26 11:59:00+00:00,600,0,no data (outage / archive gap)
3,2019-05-15 03:00:00+00:00,2019-05-15 12:59:00+00:00,600,0,no data (outage / archive gap)
4,2019-08-15 02:00:00+00:00,2019-08-15 09:59:00+00:00,480,0,no data (outage / archive gap)
5,2018-07-04 00:23:00+00:00,2018-07-04 07:59:00+00:00,457,0,no data (outage / archive gap)
6,2017-09-06 16:00:00+00:00,2017-09-06 22:59:00+00:00,420,0,no data (outage / archive gap)
7,2018-11-14 02:00:00+00:00,2018-11-14 08:59:00+00:00,420,0,no data (outage / archive gap)
8,2019-03-12 02:00:00+00:00,2019-03-12 07:59:00+00:00,360,0,no data (outage / archive gap)
9,2020-02-19 11:36:00+00:00,2020-02-19 17:29:00+00:00,354,0,no data (outage / archive gap)


In [7]:
show(SYMBOL + '_audit_outliers_top50.csv')

**BTCUSDT_audit_outliers_top50.csv**

,open_time,open,high,low,close,volume,n_trades,ret_1m,robust_z,wick_range,flag_type,ret_next_10m,volume_vs_daily_median,assessment
0,2017-08-22 05:45:00+00:00,3600.00,3600.00,3400.00,3400.00,2.020039,4,-0.073688,-736.877158,0.057158,large (>3% move or range),0.000000,8.921212,consistent with genuine market move
1,2017-08-22 05:36:00+00:00,3601.03,3601.03,3419.99,3419.99,2.948559,8,-0.073275,-732.751251,0.051582,large (>3% move or range),-0.005862,13.055444,consistent with genuine market move
2,2017-08-22 05:39:00+00:00,3665.99,3665.99,3662.00,3662.00,0.008601,4,0.068372,683.718192,0.001089,large (>3% move or range),-0.074234,0.038060,suspect (thin + full reversal)
3,2017-08-22 05:57:00+00:00,3626.99,3626.99,3626.99,3626.99,0.006525,1,0.064628,646.276716,0.000000,large (>3% move or range),0.000000,0.027885,consistent with genuine market move
4,2017-09-04 08:41:00+00:00,4299.99,4339.99,4299.99,4330.00,7.990421,19,0.059949,599.486090,0.009259,large (>3% move or range),0.005300,21.375907,consistent with genuine market move
5,2017-09-04 08:38:00+00:00,4089.99,4298.95,4075.00,4298.95,9.909321,29,0.053500,535.000601,0.053500,large (>3% move or range),0.020936,26.500329,consistent with genuine market move
6,2017-09-04 08:39:00+00:00,4162.52,4300.00,4078.05,4078.05,2.759210,13,-0.052752,-527.518738,0.052996,large (>3% move or range),0.072222,7.378908,consistent with genuine market move
7,2017-08-22 05:16:00+00:00,3599.45,3599.45,3599.45,3599.45,0.004156,1,0.042117,421.170119,0.000000,large (>3% move or range),0.026367,0.019213,consistent with genuine market move
8,2017-08-22 06:18:00+00:00,3619.95,3775.00,3619.95,3775.00,4.680001,13,0.039997,399.972795,0.041940,large (>3% move or range),0.006345,19.376399,consistent with genuine market move
9,2017-08-22 01:37:00+00:00,3844.98,3844.98,3700.00,3700.00,2.249995,24,-0.038436,-384.355817,0.038436,large (>3% move or range),0.000000,18.061659,consistent with genuine market move


In [8]:
show(SYMBOL + '_audit_file_summary.csv')

**BTCUSDT_audit_file_summary.csv**

,source_file,rows,ts_unit,has_header,first,last,expected_rows,coverage,ordered_in_file
0,BTCUSDT-1m-2017-08.zip,21360,ms,False,2017-08-17 04:00:00+00:00,2017-08-31 23:59:00+00:00,44640,0.478495,True
1,BTCUSDT-1m-2017-09.zip,42780,ms,False,2017-09-01 00:00:00+00:00,2017-09-30 23:59:00+00:00,43200,0.990278,True
2,BTCUSDT-1m-2017-10.zip,44640,ms,False,2017-10-01 00:00:00+00:00,2017-10-31 23:59:00+00:00,44640,1.000000,True
3,BTCUSDT-1m-2017-11.zip,43200,ms,False,2017-11-01 00:00:00+00:00,2017-11-30 23:59:00+00:00,43200,1.000000,True
4,BTCUSDT-1m-2017-12.zip,44564,ms,False,2017-12-01 00:00:00+00:00,2017-12-31 23:59:00+00:00,44640,0.998297,True
...,...,...,...,...,...,...,...,...,...
105,BTCUSDT-1m-2026-05.zip,44640,us,False,2026-05-01 00:00:00+00:00,2026-05-31 23:59:00+00:00,44640,1.000000,True
106,BTCUSDT-1m-2026-06.zip,43200,us,False,2026-06-01 00:00:00+00:00,2026-06-30 23:59:00+00:00,43200,1.000000,True
107,BTCUSDT-1m-2026-07.zip,44640,us,False,2026-07-01 00:00:00+00:00,2026-07-31 23:59:00+00:00,44640,1.000000,True
108,BTCUSDT-1m-2026-08.zip,44640,us,False,2026-08-01 00:00:00+00:00,2026-08-31 23:59:00+00:00,44640,1.000000,True


**Interpretation:** see the matching section of `reports/final_report.md`, written from these outputs.